# Fitted parameters: calibration and independent validation

This Notebook validates the fitted profile and does not reuse historical policy conclusions.

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd()
OUT=ROOT/'reliable-calibration'
parameters=json.loads((OUT/'parameters.json').read_text(encoding='utf-8'))
display(pd.read_csv(OUT/'parameter-comparison.csv'))

In [ ]:
display(Image(filename=str(OUT/'calibration-evidence.png')))
beta=json.loads((OUT/'beta-fit-report.json').read_text())
display(pd.DataFrame([{'profile':'original',**beta['validation_published']},{'profile':'new',**beta['validation_new']}]))
assert beta['training_criterion_passed'] and beta['validation_criteria_passed']

In [ ]:
from reliable_worker import run
case={'profile':'new','seed':1901,'n_agents':1000,'years':20,'beta_current_state':parameters['beta_current_state']}
live=run(case)
records=[json.loads(line) for line in (OUT/'beta-live.jsonl').read_text().splitlines()]
previous=next(r['result'] for r in records if r['case']==case)
difference=max(abs(live[k]-previous[k]) for k in live)
assert difference<1e-12
print('Actual model repeat maximum difference:',difference)
print('Live migration rate:',live['move_rate'])

In [ ]:
research=json.loads((OUT/'research-design-fit-report.json').read_text())
assert research['training_criterion_passed'] and research['validation_criteria_passed']
case={'profile':'new','seed':3001,'n_agents':600,'years':12,'beta_current_state':parameters['beta_by_design']['600x12']}
live=run(case)
records=[json.loads(line) for line in (OUT/'research-design-live.jsonl').read_text().splitlines()]
previous=next(r['result'] for r in records if r['case']==case)
difference=max(abs(live[k]-previous[k]) for k in live)
assert difference<1e-12
print('600x12 actual repeat maximum difference:',difference)
display(pd.DataFrame([research['validation_new']]))

In [ ]:
precision=json.loads((OUT/'precision-fix-report.json').read_text())
assert precision['passed'] and precision['mismatch_count']==0
print('Canonical precision verification:',precision['published_setting_rows'],'rows;',precision['numeric_metrics_checked'],'metrics; zero differences')

The checks demonstrate conditional simulator fit and reproducible execution. Simulation intervals exclude survey sampling and structural uncertainty. All research policy tables are separately rerun in data/research.